In [87]:
import os

In [88]:
# from models.modeling_llama import LlamaForCausalLM
# import 

# target = LlamaForCausalLM.from_pretrained("NousResearch/Yarn-Llama-2-7b-128k", cache_dir=args.cache_dir, torch_dtype=torch.float16, device_map="cuda:0")

In [89]:
import torch

def simquant(inp, bits, q_channel=-1):
    maxval = torch.max(inp, dim=q_channel).values
    minval = torch.min(inp, dim=q_channel).values
    # print('inp', inp.shape)

    rangeval = (maxval - minval)
    qx = (2**bits - 1) / rangeval
    offset = minval * qx
    offset = offset.unsqueeze(q_channel)
    qx = qx.unsqueeze(q_channel)
    # print('qx', qx.shape)
    # print('offset', offset.shape)
    qinp = torch.round(qx * inp - offset)
    qinp = torch.clip(qinp, min=0, max=2**bits - 1)
    qinp_out = (qinp + offset) / qx
    qinp_out = torch.nan_to_num(qinp_out, nan=0.0, posinf=0.0, neginf=0.0)
    return qinp_out


def simquantv2(inp, bits, q_channel=-1):
    # Makes sure that 0 can be represented
    maxval = torch.max(inp, dim=q_channel).values
    minval = torch.min(inp, dim=q_channel).values
    # print('inp', inp.shape)

    # regular quantization
    scale = (maxval - minval) / (2 ** bits - 1)
    qx = torch.clamp(torch.round((inp - minval) / scale), min=0, max=2**bits - 1)
    rqx = qx * scale + minval

    # if maxval is positive, and minval is negative, we hope zero can be exactly represented after quantization
    sign_mask = (maxval * minval) < 0

    izero_n = torch.round(minval.abs() / scale) # negative
    izero_p = torch.round(maxval.abs() / scale) # positive
    new_scale_n = minval.abs() / izero_n
    new_scale_n[izero_n == 0] = 100 # any number is OK
    new_scale_p = maxval.abs() / izero_p
    new_scale_p[izero_p == 0] = 100

    # If the negative value is too small, 
    new_scale = new_scale_n * (izero_n != 0) + new_scale_p * (izero_n == 0)
    new_offset = minval * (izero_n != 0) + (-new_scale_p) * (izero_n == 0)
    new_qx = torch.clamp(torch.round((inp - new_offset) / new_scale), min=0, max=2**bits - 1)
    new_rqx = new_qx * new_scale + new_offset

    final_rqx = rqx * (torch.logical_not(sign_mask)) + new_rqx * sign_mask
    return final_rqx

In [90]:
x = torch.arange(-3, 3)

rqx1 = simquant(x, 2, -1)
print(rqx1)


rqx2 = simquantv2(x, 2, -1)
print(rqx2)

tensor([-3.0000, -1.3333, -1.3333,  0.3333,  0.3333,  2.0000])
tensor([-3.0000, -1.5000, -1.5000,  0.0000,  1.5000,  1.5000])


In [91]:
x = torch.tensor([-0.1, 0, 1, 2, 3])

rqx1 = simquant(x, 2, -1)
print(rqx1)


rqx2 = simquantv2(x, 2, -1)
print(rqx2)


tensor([-0.1000, -0.1000,  0.9333,  1.9667,  3.0000])
tensor([0., 0., 1., 2., 2.])


In [92]:
x = torch.arange(-10, -3)

rqx1 = simquant(x, 2, -1)
print(rqx1)


rqx2 = simquantv2(x, 2, -1)
print(rqx2)

tensor([-10., -10.,  -8.,  -6.,  -6.,  -6.,  -4.])
tensor([-10., -10.,  -8.,  -6.,  -6.,  -6.,  -4.])
